In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Definir catálogo y esquemas
CATALOG = "smart_claims"
LANDING_SCHEMA = "landing"
BRONZE_SCHEMA = "bronze"
VOLUME_NAME = "landing"

# Rutas del volumen landing
COSTUMERS_PATH = f"/Volumes/{CATALOG}/{LANDING_SCHEMA}/{VOLUME_NAME}/customers/customers.csv"
POLICIES_PATH = f"/Volumes/{CATALOG}/{LANDING_SCHEMA}/{VOLUME_NAME}/policies/policies.csv"
CLAIMS_PATH = f"/Volumes/{CATALOG}/{LANDING_SCHEMA}/{VOLUME_NAME}/claims/claims.csv"
TELEMATICS_PATH = f"/Volumes/{CATALOG}/{LANDING_SCHEMA}/{VOLUME_NAME}/telematics/"
TRAINING_IMGS_PATH = f"/Volumes/{CATALOG}/{LANDING_SCHEMA}/{VOLUME_NAME}/training_imgs/"
CLAIM_IMAGES_PATH = f"/Volumes/{CATALOG}/{LANDING_SCHEMA}/{VOLUME_NAME}/claim_images/"
CLAIM_IMAGES_METADATA_PATH = f"/Volumes/{CATALOG}/{LANDING_SCHEMA}/{VOLUME_NAME}/claim_images_metadata/image_metadata.csv"

print("✓ Variables configuradas")
print(f"Catálogo: {CATALOG}")
print(f"Esquema Bronze: {BRONZE_SCHEMA}")

✓ Variables configuradas
Catálogo: smart_claims
Esquema Bronze: bronze


In [0]:
# Leer archivo CSV de customers
df_customers = (spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(COSTUMERS_PATH)
)

# Guardar como tabla Delta en bronze
df_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.customers")

print(f"✓ Tabla {CATALOG}.{BRONZE_SCHEMA}.customers creada")
print(f"  Registros: {df_customers.count()}")

✓ Tabla smart_claims.bronze.customers creada
  Registros: 7061


In [0]:
# Leer archivo CSV de policies
df_policies = (spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(POLICIES_PATH)
)

# Guardar como tabla Delta en bronze
df_policies.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.policies")

print(f"✓ Tabla {CATALOG}.{BRONZE_SCHEMA}.policies creada")
print(f"  Registros: {df_policies.count()}")

✓ Tabla smart_claims.bronze.policies creada
  Registros: 12237


In [0]:
# Leer archivo CSV de claims
df_claims = (spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(CLAIMS_PATH)
)

# Guardar como tabla Delta en bronze
df_claims.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.claims")

print(f"✓ Tabla {CATALOG}.{BRONZE_SCHEMA}.claims creada")
print(f"  Registros: {df_claims.count()}")

✓ Tabla smart_claims.bronze.claims creada
  Registros: 12991


In [0]:
# Leer carpeta de archivos Parquet de telematics
df_telematics = spark.read \
    .format("parquet") \
    .load(TELEMATICS_PATH)

# Guardar como tabla Delta en bronze
df_telematics.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.telematics")

print(f"✓ Tabla {CATALOG}.{BRONZE_SCHEMA}.telematics creada")
print(f"  Registros: {df_telematics.count()}")

✓ Tabla smart_claims.bronze.telematics creada
  Registros: 780060


In [0]:
# Leer imágenes de training como archivos binarios
df_training_images = spark.read \
    .format("binaryFile") \
    .load(TRAINING_IMGS_PATH)

# Guardar como tabla Delta en bronze (si hay imágenes)
if df_training_images.count() > 0:
    df_training_images.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.training_images")
    print(f"✓ Tabla {CATALOG}.{BRONZE_SCHEMA}.training_images creada")
    print(f"  Registros: {df_training_images.count()}")
else:
    print(f"⚠️  Carpeta training_imgs está vacía - tabla NO creada")

✓ Tabla smart_claims.bronze.training_images creada
  Registros: 56


In [0]:
# Leer imágenes de claims como archivos binarios
df_claim_images = spark.read \
    .format("binaryFile") \
    .load(CLAIM_IMAGES_PATH)

# Guardar como tabla Delta en bronze
df_claim_images.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.claim_images")

print(f"✓ Tabla {CATALOG}.{BRONZE_SCHEMA}.claim_images creada")
print(f"  Registros: {df_claim_images.count()}")

✓ Tabla smart_claims.bronze.claim_images creada
  Registros: 15


In [0]:
# Leer archivo CSV de metadata de imágenes
df_claim_images_metadata = (spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(CLAIM_IMAGES_METADATA_PATH)
)

# Guardar como tabla Delta en bronze
df_claim_images_metadata.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.claim_images_metadata")

print(f"✓ Tabla {CATALOG}.{BRONZE_SCHEMA}.claim_images_metadata creada")
print(f"  Registros: {df_claim_images_metadata.count()}")
print("  Columnas:", df_claim_images_metadata.columns)

✓ Tabla smart_claims.bronze.claim_images_metadata creada
  Registros: 13001
  Columnas: ['image_name', 'image_id', 'claim_no', 'chassis_no']


In [0]:
print("===" * 20)
print("VERIFICACIÓN DE CAPA BRONZE")
print("===" * 20)

# Listar todas las tablas en el esquema bronze
print(f"\n📋 Tablas en {CATALOG}.{BRONZE_SCHEMA}:\n")
tablas_bronze = spark.sql(f"SHOW TABLES IN {CATALOG}.{BRONZE_SCHEMA}").collect()

if len(tablas_bronze) == 0:
    print("  ⚠️  No hay tablas en el esquema bronze aún.")
    print("  👉 Ejecuta las celdas 2-8 primero para crear las tablas.")
else:
    for tabla in tablas_bronze:
        nombre_tabla = tabla['tableName']
        
        # Contar registros de cada tabla
        count = spark.sql(f"SELECT COUNT(*) as total FROM {CATALOG}.{BRONZE_SCHEMA}.{nombre_tabla}").collect()[0]['total']
        
        print(f"  ✓ {nombre_tabla:<30} {count:>10,} registros")
    
    # Verificar columnas de claim_images_metadata si existe
    if any(t['tableName'] == 'claim_images_metadata' for t in tablas_bronze):
        print(f"\n🔍 Detalle de claim_images_metadata:")
        df_metadata = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.claim_images_metadata")
        print(f"  Columnas: {', '.join(df_metadata.columns)}")
    
    print("\n" + "===" * 20)
    print("✓ CAPA BRONZE COMPLETADA")
    print("===" * 20)

VERIFICACIÓN DE CAPA BRONZE

📋 Tablas en smart_claims.bronze:

  ✓ claim_images                           15 registros
  ✓ claim_images_metadata              13,001 registros
  ✓ claims                             12,991 registros
  ✓ customers                           7,061 registros
  ✓ policies                           12,237 registros
  ✓ telematics                        780,060 registros

🔍 Detalle de claim_images_metadata:
  Columnas: image_name, image_id, claim_no, chassis_no

✓ CAPA BRONZE COMPLETADA


In [0]:
# CUSTOMERS - Datos personales del cliente
df_customers_exp = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers")

print("🔍 BRONZE.CUSTOMERS - Datos personales del cliente")
print(f"Total registros: {df_customers_exp.count():,}")
print(f"Columnas: {df_customers_exp.columns}\n")

display(df_customers_exp.limit(10))

🔍 BRONZE.CUSTOMERS - Datos personales del cliente
Total registros: 7,061
Columnas: ['customer_id', 'date_of_birth', 'borough', 'neighborhood', 'zip_code', 'name']



customer_id,date_of_birth,borough,neighborhood,zip_code,name
104.0,null,BROOKLYN,NORTHWEST BROOKLYN,11215,"Schmitt, Marlene"
106.0,01-01-1999,BROOKLYN,FLATBUSH,11203,"Engel, Lara"
111.0,null,MANHATTAN,GRAMERCY PARK AND MURRAY HILL,10017,"Sauer, Noah"
112.0,01-01-2000,MANHATTAN,CHELSEA AND CLINTON,10019,"Michel, Mila"
113.0,18-05-2009,STATEN ISLAND,STAPLETON AND ST. GEORGE,10301,"Seidel, Leni"
115.0,16-05-2005,QUEENS,CENTRAL QUEENS,11365,"Schreiber, Laurin"
117.0,12-11-2010,QUEENS,SOUTHEAST QUEENS,11005,"Schuster, Frieda"
128.0,23-10-1997,MANHATTAN,CENTRAL HARLEM,10027,"Stephan, Alexander"
130.0,12-11-2010,STATEN ISLAND,PORT RICHMOND,10303,"Schilling, Leni"
136.0,01-01-1999,BROOKLYN,SOUTHERN BROOKLYN,11229,"Klein, Malte"


In [0]:
# POLICIES - Información de póliza y vehículo
df_policies_exp = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.policies")

print("🔍 BRONZE.POLICIES - Información de póliza y vehículo")
print(f"Total registros: {df_policies_exp.count():,}")
print(f"Columnas: {df_policies_exp.columns}\n")

display(df_policies_exp.limit(10))

🔍 BRONZE.POLICIES - Información de póliza y vehículo
Total registros: 12,237
Columnas: ['POLICY_NO', 'CUST_ID', 'POLICYTYPE', 'POL_ISSUE_DATE', 'POL_EFF_DATE', 'POL_EXPIRY_DATE', 'MAKE', 'MODEL', 'MODEL_YEAR', 'CHASSIS_NO', 'USE_OF_VEHICLE', 'PRODUCT', 'SUM_INSURED', 'PREMIUM', 'DEDUCTABLE']



POLICY_NO,CUST_ID,POLICYTYPE,POL_ISSUE_DATE,POL_EFF_DATE,POL_EXPIRY_DATE,MAKE,MODEL,MODEL_YEAR,CHASSIS_NO,USE_OF_VEHICLE,PRODUCT,SUM_INSURED,PREMIUM,DEDUCTABLE
102122649,9990.0,COMP,2018-09-16,2018-09-16,2019-10-15,RENAULT,MEGANE,2015.0,VF1BZBCT7FR517802,PRIVATE,STANDARD,32000.0,1460.0,1000
102123206,8825.0,COMP,2018-10-02,2018-10-02,2019-11-01,PEUGEOT,BOXER,2017.0,VF3YCZMF8H2D49283,COMMERCIAL,STANDARD,107200.0,3654.0,1000
102123347,243.0,COMP,2018-10-08,2018-11-12,2019-12-11,TOYOTA,HIACE,2017.0,JTFPX22P2H0077536,COMMERCIAL,STANDARD,71200.0,2552.0,500
102123550,4500.0,COMP,2018-10-14,2018-11-28,2019-12-27,NISSAN,TIIDA,2011.0,3N1BC1C63BK196312,PRIVATE,STANDARD,9000.0,1460.0,500
102128327,9244.0,COMP,2019-03-05,2019-04-19,2020-05-18,NISSAN,SUNNY,2015.0,MDHBN7AD3FG801977,PRIVATE,STANDARD,17520.0,1190.0,2000
102128941,903.0,COMP,2019-03-20,2019-03-21,2020-04-20,AUDI,Q 7,2013.0,WA1AGDFE9DD010044,PRIVATE,M 2.5,81752.0,2295.0,500
102128971,5172.0,COMP,2019-03-21,2019-03-25,2020-04-24,HONDA,CIVIC,2008.0,JHMFD16288S419306,PRIVATE,NOT CLASSIFIED,8000.0,1190.0,2000
102130296,7848.0,COMP,2019-05-05,2019-05-05,2020-06-04,PEUGEOT,3008 GT,2019.0,VF3M45GY7KS013484,PRIVATE,M 2.5,118000.0,3001.0,500
102130827,12714.0,COMP,2019-05-26,2019-05-26,2020-06-25,MG,ZS,2019.0,LSJW74U6XKZ012004,PRIVATE,M 2.5,54650.0,1940.0,500
102131025,7957.0,COMP,2019-05-29,2019-05-29,2020-06-28,NISSAN,ALTIMA,2016.0,1N4AL3A9XGC104939,PRIVATE,M 2019,35926.0,1210.0,1000


In [0]:
# CLAIMS - Información del reclamo
df_claims_exp = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.claims")

print("🔍 BRONZE.CLAIMS - Información del reclamo")
print(f"Total registros: {df_claims_exp.count():,}")
print(f"Columnas: {df_claims_exp.columns}\n")

display(df_claims_exp.limit(10))

🔍 BRONZE.CLAIMS - Información del reclamo
Total registros: 12,991
Columnas: ['claim_no', 'policy_no', 'claim_date', 'months_as_customer', 'injury', 'property', 'vehicle', 'total', 'collision_type', 'number_of_vehicles_involved', 'age', 'insured_relationship', 'license_issue_date', 'date', 'hour', 'type', 'severity', 'number_of_witnesses', 'suspicious_activity']



claim_no,policy_no,claim_date,months_as_customer,injury,property,vehicle,total,collision_type,number_of_vehicles_involved,age,insured_relationship,license_issue_date,date,hour,type,severity,number_of_witnesses,suspicious_activity
f59f665f-c82f-4aa3-98b5-060c7a8f6d19,102049729,2016-01-26,259,780,780,3510,5070,Side Collision,4,40.0,unmarried,2006-05-24,2016-10-14,10,Multi-vehicle Collision,Minor Damage,0,false
f051f2ae-c92e-44f9-9bb8-3ab60b19341b,102147750,2020-08-18,261,9680,14520,38720,62920,Side Collision,4,24.0,other-relative,2016-10-25,2021-05-09,7,Multi-vehicle Collision,Major Damage,2,false
1aec2184-3a54-4b3b-a41b-51b79feb4538,102139065,2020-01-20,86,4060,4060,24360,32480,Rear Collision,1,32.0,own-child,2008-12-22,2020-10-11,3,Parked Car,Total Loss,0,false
22b8e795-10e8-4981-b62c-59607a215655,102099757,2017-04-12,121,670,670,4690,6030,Rear Collision,1,23.0,other-relative,2012-08-12,2017-11-27,21,Parked Car,Total Loss,0,false
5ccc8c8f-d199-4cad-af49-90e45a729489,102071474,2017-01-22,215,15620,7810,54670,78100,Front Collision,1,26.0,husband,2012-08-08,2017-05-21,9,Single Vehicle Collision,Major Damage,3,true
7730a464-78e3-49ee-98b4-40072c13f492,102136098,2019-10-14,290,4570,4570,31990,41130,Front Collision,4,27.0,unmarried,2014-01-26,2020-03-30,19,Multi-vehicle Collision,Major Damage,2,false
6bbca89a-4c02-4beb-ab06-63d301f6f39e,102106062,2017-06-17,371,4730,4730,37840,47300,Rear Collision,1,30.0,not-in-family,2010-01-07,2018-02-03,2,Single Vehicle Collision,Major Damage,0,true
25144193-d014-44d7-ad7d-771337bd294c,102131860,2019-07-06,338,5190,10380,36330,51900,Side Collision,4,37.0,unmarried,2016-05-01,2020-02-22,21,Multi-vehicle Collision,Minor Damage,3,true
fae3651c-f0c8-421b-bbbe-3483d6c03def,102097981,2017-04-12,190,4940,9880,34580,49400,Front Collision,4,23.0,wife,2016-11-29,2017-12-03,22,Multi-vehicle Collision,Minor Damage,1,false
c0c6bd09-de4c-4697-8772-d55629f81b78,102076310,2017-03-13,280,5350,5350,42800,53500,Side Collision,4,38.0,not-in-family,2008-06-11,2017-10-31,4,Multi-vehicle Collision,Total Loss,2,false


In [0]:
# TELEMATICS - Datos de velocidad, coordenadas y tiempo
df_telematics_exp = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.telematics")

print("🔍 BRONZE.TELEMATICS - Datos de velocidad, coordenadas GPS y tiempo")
print(f"Total registros: {df_telematics_exp.count():,}")
print(f"Columnas: {df_telematics_exp.columns}\n")

display(df_telematics_exp.limit(10))

🔍 BRONZE.TELEMATICS - Datos de velocidad, coordenadas GPS y tiempo
Total registros: 780,060
Columnas: ['chassis_no', 'latitude', 'longitude', 'event_timestamp', 'speed']



chassis_no,latitude,longitude,event_timestamp,speed
YV2RS02D7EA767675,40.80879,-73.916184,2015-12-04 11:09:00,46.78709030151367
YV2RS02D7EA767675,40.801766,-73.937235,2015-12-04 11:08:00,55.857643127441406
YV2RS02D7EA767675,40.802891,-73.933859,2015-12-04 11:07:00,9.668610572814941
YV2RS02D7EA767675,40.802951,-73.934006,2015-12-04 11:06:00,10.625295639038086
YV2RS02D7EA767675,40.803159,-73.934536,2015-12-04 11:05:00,51.81836700439453
YV2RS02D7EA767675,40.803641,-73.935677,2015-12-04 11:04:00,49.12469482421875
YV2RS02D7EA767675,40.803706,-73.935829,2015-12-04 11:03:00,49.15388107299805
YV2RS02D7EA767675,40.803611,-73.935897,2015-12-04 11:02:00,54.84403991699219
YV2RS02D7EA767675,40.802397,-73.93677,2015-12-04 11:01:00,26.55282211303711
YV2RS02D7EA767675,40.801143,-73.937694,2015-12-04 11:00:00,16.455142974853516


In [0]:
# CLAIM_IMAGES - Ruta del archivo y contenido binario
df_claim_images_exp = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.claim_images")

print("🔍 BRONZE.CLAIM_IMAGES - Ruta y contenido binario de imágenes")
print(f"Total registros: {df_claim_images_exp.count():,}")
print(f"Columnas: {df_claim_images_exp.columns}")
print("  - path: Ruta del archivo de imagen")
print("  - modificationTime: Fecha de modificación")
print("  - length: Tamaño del archivo en bytes")
print("  - content: Contenido binario de la imagen\n")

# Mostrar solo path, length, modificationTime (sin el contenido binario completo)
display(df_claim_images_exp.select("path", "length", "modificationTime"))

🔍 BRONZE.CLAIM_IMAGES - Ruta y contenido binario de imágenes
Total registros: 15
Columnas: ['path', 'modificationTime', 'length', 'content']
  - path: Ruta del archivo de imagen
  - modificationTime: Fecha de modificación
  - length: Tamaño del archivo en bytes
  - content: Contenido binario de la imagen



path,length,modificationTime
dbfs:/Volumes/smart_claims/landing/landing/claim_images/1_High.jpg,634689,2026-04-11T13:57:40.000Z
dbfs:/Volumes/smart_claims/landing/landing/claim_images/1_Low.jpg,488305,2026-04-11T13:57:39.000Z
dbfs:/Volumes/smart_claims/landing/landing/claim_images/1_Medium.jpg,595993,2026-04-11T13:57:40.000Z
dbfs:/Volumes/smart_claims/landing/landing/claim_images/2_High.jpg,1928661,2026-04-11T13:57:45.000Z
dbfs:/Volumes/smart_claims/landing/landing/claim_images/2_Low.jpg,1469269,2026-04-11T13:57:44.000Z
dbfs:/Volumes/smart_claims/landing/landing/claim_images/2_Medium.jpg,1603316,2026-04-11T13:57:44.000Z
dbfs:/Volumes/smart_claims/landing/landing/claim_images/3_High.jpg,1748114,2026-04-11T13:57:44.000Z
dbfs:/Volumes/smart_claims/landing/landing/claim_images/3_Low.jpg,1583862,2026-04-11T13:57:44.000Z
dbfs:/Volumes/smart_claims/landing/landing/claim_images/3_Medium.jpg,1602524,2026-04-11T13:57:44.000Z
dbfs:/Volumes/smart_claims/landing/landing/claim_images/4_High.jpg,1844884,2026-04-11T13:57:45.000Z


In [0]:
# CLAIM_IMAGES_METADATA - Relación entre imagen y reclamo
df_metadata_exp = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.claim_images_metadata")

print("🔍 BRONZE.CLAIM_IMAGES_METADATA - Relación entre imagen y reclamo")
print(f"Total registros: {df_metadata_exp.count():,}")
print(f"Columnas: {df_metadata_exp.columns}")
print("  - image_name: Nombre del archivo de imagen")
print("  - image_id: ID único de la imagen")
print("  - claim_no: Número de reclamo asociado")
print("  - chassis_no: Número de chasis del vehículo\n")

display(df_metadata_exp.limit(10))

🔍 BRONZE.CLAIM_IMAGES_METADATA - Relación entre imagen y reclamo
Total registros: 13,001
Columnas: ['image_name', 'image_id', 'claim_no', 'chassis_no']
  - image_name: Nombre del archivo de imagen
  - image_id: ID único de la imagen
  - claim_no: Número de reclamo asociado
  - chassis_no: Número de chasis del vehículo



image_name,image_id,claim_no,chassis_no
4_High.jpg,10,32d5b8fd-ed32-4370-aac3-9e7d8fcb499d,JN8BT05Y87W111874
4_High.jpg,10,1ee11edb-6169-4c24-80f8-110461f59777,6G1EK54HX9L237163
4_High.jpg,10,27cfb515-af2f-4b91-a4b5-9c57836879af,6G1EK54HX9L237163
4_High.jpg,10,05e83630-4488-441a-a566-953384630511,5N1AL0MM7DC333224
4_High.jpg,10,4abc9897-e0a4-4705-83b5-4fddd707b0fc,JTHBG262582012524
4_High.jpg,10,5eabe7d8-9d70-4406-a815-4ac38234581e,JN1FN61C08W093833
4_High.jpg,10,9d011028-c7e1-4aa7-83af-4de41c98ccd8,JNRAR07Y7XW066462
4_High.jpg,10,091ff093-4291-44e1-a614-58603b16f91b,5N1AR2M51FC631760
4_High.jpg,10,b6d72036-8cb2-4c7b-87c9-cf75163b95b3,JTDBW923794025112...
4_High.jpg,10,42053276-1b49-4412-8c51-86f278b38fd2,JTDBW923794025112...


In [0]:
print("="*60)
print("CONCLUSIONES - CAPA BRONZE")
print("="*60)

print("\n👉 QUÉ ES LA CAPA BRONZE:")
print("  La capa Bronze es el PRIMER ATERRIZAJE de los datos en Unity Catalog.")
print("  Conserva los datos CASI COMO LLEGAN, sin transformaciones.")

print("\n✅ QUÉ CONTIENE:")
print("  • customers: Datos personales (7,061 registros)")
print("  • policies: Pólizas y vehículos (12,237 registros)")
print("  • claims: Información de reclamos (12,991 registros)")
print("  • telematics: GPS, velocidad, timestamps (780,060 registros)")
print("  • claim_images: Imágenes binarias (15 archivos)")
print("  • claim_images_metadata: Relación imagen-reclamo (13,001 registros)")

print("\n⚠️  ESTADO DE LOS DATOS:")
print("  Pueden existir:")
print("  • Inconsistencias de formato")
print("  • Nombres de columnas sin estandarizar")
print("  • Tipos de dato incorrectos")
print("  • Valores nulos o duplicados")
print("  • Fechas en formato string en lugar de timestamp")

print("\n🚀 PRÓXIMO PASO:")
print("  La CAPA SILVER aplicará limpieza y transformaciones:")
print("  • Estandarizar nombres de columnas")
print("  • Convertir tipos de dato correctos")
print("  • Manejar valores nulos")
print("  • Eliminar duplicados")
print("  • Aplicar reglas de negocio")

print("\n" + "="*60)